In [5]:
!pip install flash_attn

  Using cached flash_attn-2.8.2.tar.gz (8.2 MB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Using cached einops-0.8.1-py3-none-any.whl.metadata (13 kB)
Using cached einops-0.8.1-py3-none-any.whl (64 kB)
  Running setup.py clean for flash_attn
Failed to build flash_attn


  DEPRECATION: Building 'flash_attn' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'flash_attn'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  error: subprocess-exited-with-error
  
  python setup.py bdist_wheel did not run successfully.
  exit code: 1
  
  [177 lines of output]
  C:\Users\karol\miniconda3\envs\scraper_conda\lib\site-packages\torch\utils\cpp_extension.py:28: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
    from pkg_resources import packaging  # type: ignore[a

In [ ]:
w =4

In [4]:
!pip install torch==2.1.2+cu118 torchvision==0.16.2+cu118 torchaudio==2.1.2 -f https://download.pytorch.org/whl/cu118/torch_stable.html

Looking in links: https://download.pytorch.org/whl/cu118/torch_stable.html
     ---------------------------------------- 0.0/2.7 GB ? eta -:--:--
     ---------------------------------------- 0.0/2.7 GB 56.4 MB/s eta 0:00:49
     ---------------------------------------- 0.0/2.7 GB 83.8 MB/s eta 0:00:33
      --------------------------------------- 0.0/2.7 GB 86.2 MB/s eta 0:00:32
     - -------------------------------------- 0.1/2.7 GB 91.9 MB/s eta 0:00:29
     - -------------------------------------- 0.1/2.7 GB 100.6 MB/s eta 0:00:27
     - -------------------------------------- 0.1/2.7 GB 103.9 MB/s eta 0:00:25
     -- ------------------------------------- 0.2/2.7 GB 105.4 MB/s eta 0:00:25
     -- ------------------------------------- 0.2/2.7 GB 107.4 MB/s eta 0:00:24
     -- ------------------------------------- 0.2/2.7 GB 107.0 MB/s eta 0:00:24
     --- ------------------------------------ 0.2/2.7 GB 103.9 MB/s eta 0:00:25
     --- ------------------------------------ 0.2/2.7 GB 1

  You can safely remove it manually.


In [3]:
import torch

print(torch.cuda.is_available())

True


In [1]:
from preprocessing.utils.defaults import AWS_REGION
import sagemaker
import boto3

boto_sess = boto3.Session(region_name=AWS_REGION)

# 3. SageMaker session that uses the same region
sess = sagemaker.Session(boto_session=boto_sess)



sagemaker_session_bucket=None
if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = sagemaker.get_execution_role()
except ValueError:
    iam = boto3.client('iam')
    role = iam.get_role(RoleName='sagemaker_execution_role')['Role']['Arn']

sess = sagemaker.Session(boto_session=boto3.Session(region_name=AWS_REGION), default_bucket=sagemaker_session_bucket)

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

sagemaker.config INFO - Not applying SDK defaults from location: C:\ProgramData\sagemaker\sagemaker\config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: C:\Users\karol\AppData\Local\sagemaker\sagemaker\config.yaml


Couldn't call 'get_role' to get Role ARN from role name superUser to get Role path.


sagemaker role arn: arn:aws:iam::767427092061:role/sagemaker_execution_role
sagemaker bucket: sagemaker-eu-west-1-767427092061
sagemaker session region: eu-west-1


In [2]:
from datasets import load_dataset

# 0️⃣  Config -------------------------------------------------------------------
query_prefix = "[query]: "                  # or pull from argparse / env

def add_query_prefix(example):
    example["anchor"] = f"{query_prefix}{example['anchor']}"
    return example

# Load dataset from the hub
dataset = load_dataset("philschmid/finanical-rag-embedding-dataset", split="train")
input_path = f's3://{sess.default_bucket()}/datasets/rag-embedding'

# rename columns
dataset = dataset.rename_column("question", "anchor")
dataset = dataset.rename_column("context", "positive")

dataset = dataset.map(add_query_prefix, desc="add [query]: prefix")

# Add an id column to the dataset
dataset = dataset.add_column("id", range(len(dataset)))

# split dataset into a 10% test set
dataset = dataset.train_test_split(test_size=0.1)

# save train_dataset to s3 using our SageMaker session

# save datasets to s3
dataset["train"].to_json(f"{input_path}/train/dataset.json", orient="records")
train_dataset_s3_path = f"{input_path}/train/dataset.json"
dataset["test"].to_json(f"{input_path}/test/dataset.json", orient="records")
test_dataset_s3_path = f"{input_path}/test/dataset.json"

print(f"Training data uploaded to:")
print(train_dataset_s3_path)
print(test_dataset_s3_path)
print(f"https://s3.console.aws.amazon.com/s3/buckets/{sess.default_bucket()}/?region={sess.boto_region_name}&prefix={input_path.split('/', 3)[-1]}/")

C:\Users\karol\miniconda3\envs\scraper_conda\lib\site-packages\fsspec\registry.py:273: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


Creating json from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Training data uploaded to:
s3://sagemaker-eu-west-1-767427092061/datasets/rag-embedding/train/dataset.json
s3://sagemaker-eu-west-1-767427092061/datasets/rag-embedding/test/dataset.json
https://s3.console.aws.amazon.com/s3/buckets/sagemaker-eu-west-1-767427092061/?region=eu-west-1&prefix=datasets/rag-embedding/


In [4]:
s3_train_path = "s3://sagemaker-eu-west-1-767427092061/datasets/rag-embedding/train/dataset.json"

sample_ds = load_dataset(
    "json",
    data_files={"train": s3_train_path},   # ← wrap in dict
    split="train"          # read just the first 5 rows
)
for row in sample_ds[:5]:
    print(row["anchor"])       # ← should start with "[query]: "

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:3                                                                                    │
│                                                                                                  │
│    1 s3_train_path = "s3://sagemaker-eu-west-1-767427092061/datasets/rag-embedding/train/data    │
│    2                                                                                             │
│ ❱  3 sample_ds = load_dataset(                                                                   │
│    4 │   "json",                                                                                 │
│    5 │   data_files={"train": s3_train_path},   # ← wrap in dict                                 │
│    6 │   split="train"          # read just the first 5 rows                                     │
│                                                                                                  │
│ C:\Users\karol\miniconda3\envs\scraper_conda\lib\site-packages\datasets\load.py:2582 in          │
│ load_dataset                                                                                     │
│                                                                                                  │
│   2579 │   try_from_hf_gcs = path not in _PACKAGED_DATASETS_MODULES                              │
│   2580 │                                                                                         │
│   2581 │   # Download and prepare data                                                           │
│ ❱ 2582 │   builder_instance.download_and_prepare(                                                │
│   2583 │   │   download_config=download_config,                                                  │
│   2584 │   │   download_mode=download_mode,                                                      │
│   2585 │   │   verification_mode=verification_mode,                                              │
│                                                                                                  │
│ C:\Users\karol\miniconda3\envs\scraper_conda\lib\site-packages\datasets\builder.py:1005 in       │
│ download_and_prepare                                                                             │
│                                                                                                  │
│   1002 │   │   │   │   │   │   │   prepare_split_kwargs["max_shard_size"] = max_shard_size       │
│   1003 │   │   │   │   │   │   if num_proc is not None:                                          │
│   1004 │   │   │   │   │   │   │   prepare_split_kwargs["num_proc"] = num_proc                   │
│ ❱ 1005 │   │   │   │   │   │   self._download_and_prepare(                                       │
│   1006 │   │   │   │   │   │   │   dl_manager=dl_manager,                                        │
│   1007 │   │   │   │   │   │   │   verification_mode=verification_mode,                          │
│   1008 │   │   │   │   │   │   │   **prepare_split_kwargs,                                       │
│                                                                                                  │
│ C:\Users\karol\miniconda3\envs\scraper_conda\lib\site-packages\datasets\builder.py:1078 in       │
│ _download_and_prepare                                                                            │
│                                                                                                  │
│   1075 │   │   # Generating data for all splits                                                  │
│   1076 │   │   split_dict = SplitDict(dataset_name=self.dataset_name)                            │
│   1077 │   │   split_generators_kwargs = self._make_split_generators_kwargs(prepare_split_kwarg  │
│ ❱ 1078 │   │   split_generators = self._split_generators(dl_manager, **split_generators_kwargs)  │
│   1079 │   │                                               

In [10]:
import pandas as pd   # s3fs is auto-used under the hood

df = pd.read_json(
    "s3://sagemaker-eu-west-1-767427092061/datasets/rag-embedding/train/dataset.json",
    lines=True          # each line is one JSON record (ND-JSON)
)
print(df.head())

                                              anchor  \
0  [query]: How much growth did the Automotive se...   
1  [query]: What significant proposal did the CFP...   
2  [query]: By what percentage did HIV product sa...   
3  [query]: What are some factors that lead to a ...   
4  [query]: What was the company's strategy regar...   

                                            positive    id  
0  Automotive revenue for fiscal year 2023 grew 6...    63  
1  In February 2023, the CFPB proposed a rule tha...  6727  
2  HIV product sales increased 6% to $18.2 billio...  5467  
3  Pressures from private insurers and government...  5440  
4  In the United States, the Company has establis...  3205  


In [ ]:
import time
from sagemaker.huggingface import HuggingFace

job_name = f'huggingface-fsdp-{time.strftime("%Y-%m-%d-%H-%M-%S", time.localtime())}'

# hyperparameters, which are passed into the training job

hyperparameters={
    "model_id": "BAAI/bge-base-en-v1.5", # model id from the hub
    "train_dataset_path": "/opt/ml/input/data/train/", # path inside the container where the training data is stored
    "test_dataset_path": "/opt/ml/input/data/test/", # path inside the container where the test data is stored
    "num_train_epochs": 3, # number of training epochs
    "learning_rate": 2e-5, # learning rate
    'gradient_checkpointing': True, # enable gradient checkpointing
    'optimizer': "adamw_apex_fused", # optimizer
    'per_device_train_batch_size': 8, # batch size per device during training
    'per_device_eval_batch_size': 4,
    'fsdp': '"full_shard auto_wrap"', # fully sharded data parallelism
    'fsdp_transformer_layer_cls_to_wrap': "BertLayer", # transformer layer to wrap
    'matryoshka_dims': [1024, 768, 512, 256, 128]
}

# estimator
huggingface_estimator_distribution = HuggingFace(
    entry_point='run_clm.py',
    source_dir='./fsdp_script',
    # instance_type='ml.g5.xlarge', # 1 GPU
    instance_type='ml.g5.12xlarge', # 4 GPU
    instance_count=1,
    volume_size=200,
    role=role,
    job_name=job_name,
    transformers_version = '4.36.0',          # the transformers version used in the training job
    pytorch_version      = '2.1.0',           # the pytorch_version version used in the training job
    py_version           = 'py310',
    hyperparameters = hyperparameters,
    distribution={"torch_distributed": {"enabled": True}} # enable torchrun
)

In [3]:
r =4

In [11]:
training_arguments = {
    "model_id": "sdadas/mmlw-retrieval-roberta-large-v2", # model id from the hub
    "train_dataset_path": "/opt/ml/input/data/train/", # path inside the container where the training data is stored
    "test_dataset_path": "/opt/ml/input/data/test/", # path inside the container where the test data is stored
    "num_train_epochs": 3, # number of training epochs
    "learning_rate": 2e-5, # learning rate
    'per_device_train_batch_size': 6, # batch size per device during training
    'per_device_eval_batch_size': 4,
    'gradient_accumulation_steps':8,
    'matryoshka_dims': [1024, 768, 512, 256, 128]
}

job_name = f'roberta-large-{time.strftime("%Y-%m-%d-%H-%M-%S", time.localtime())}'

# create the Estimator
huggingface_estimator = HuggingFace(
    entry_point          = 'run_mnr.py',      # train script
    source_dir           = 'scripts_pirb',         # directory which includes all the files needed for training
    instance_type        = 'ml.g5.xlarge',    # instances type used for the training job
    instance_count       = 1,                 # the number of instances used for training
    max_run              = 2*24*60*60,        # maximum runtime in seconds (days * hours * minutes * seconds)
    base_job_name        = job_name,          # the name of the training job
    role                 = role,              # Iam role used in training job to access AWS ressources, e.g. S3
    transformers_version = '4.36.0',          # the transformers version used in the training job
    pytorch_version      = '2.1.0',           # the pytorch_version version used in the training job
    py_version           = 'py310',           # the python version used in the training job
    hyperparameters      =  training_arguments,
    disable_output_compression = True,        # not compress output to save training time and cost
    environment  = {
        "HUGGINGFACE_HUB_CACHE": "/tmp/.cache", # set env variable to cache models in /tmp
    },
)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:13                                                                                   │
│                                                                                                  │
│   10 │   'matryoshka_dims': [1024, 768, 512, 256, 128]                                           │
│   11 }                                                                                           │
│   12                                                                                             │
│ ❱ 13 job_name = f'roberta-large-{time.strftime("%Y-%m-%d-%H-%M-%S", time.localtime())}'          │
│   14                                                                                             │
│   15 # create the Estimator                                                                      │
│   16 huggingface_estimator = HuggingFace(                                                        │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
NameError: name 'time' is not defined

In [15]:
data = {
    'train': train_dataset_s3_path,
    'test': test_dataset_s3_path,
}

# starting the train job with our uploaded datasets as input
huggingface_estimator.fit(data, wait=True)
r = 4

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:7                                                                                    │
│                                                                                                  │
│   4 }                                                                                            │
│   5                                                                                              │
│   6 # starting the train job with our uploaded datasets as input                                 │
│ ❱ 7 huggingface_estimator.fit(data, wait=True)                                                   │
│   8 r = 4                                                                                        │
│   9                                                                                              │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
NameError: name 'huggingface_estimator' is not defined